# 01 · Baseline phishing URL classifier

Goal for today: load a real labeled dataset, look at it, and train two classic models on **lexical URL features only** (nothing that requires visiting the page). We report **precision and recall on the phishing class**, because accuracy hides the errors that matter.

- **Recall** = of the actual phishing URLs, how many did we catch? A miss means someone clicks.
- **Precision** = of the URLs we flagged, how many were really phishing? Low precision means users learn to ignore warnings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.parse import urlsplit

from phishdetect.data import load_phiusiil
from phishdetect.features import featurize, FEATURE_NAMES

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

## 1. Load the data
First run downloads ~235k URLs from UCI and caches them to `data/raw/phiusiil.csv` (git-ignored).

In [ ]:
df = load_phiusiil()
df = df.drop_duplicates("url").reset_index(drop=True)
print(df.shape)
df.sample(8, random_state=RANDOM_STATE)

## 2. Explore

In [ ]:
balance = df["is_phishing"].value_counts().rename({0: "legit", 1: "phishing"})
print(balance, "\n")
print((balance / len(df)).round(3))

In [ ]:
df["url_len"] = df["url"].str.len()
fig, ax = plt.subplots(figsize=(8, 4))
for label, name in [(0, "legit"), (1, "phishing")]:
    df.loc[df.is_phishing == label, "url_len"].clip(upper=200).plot.hist(
        bins=60, alpha=0.5, ax=ax, label=name, density=True)
ax.set_xlabel("URL length (clipped at 200)")
ax.legend(); plt.show()
df.groupby("is_phishing")["url_len"].describe()

In [ ]:
def host_of(u):
    u = u if "://" in u else "http://" + u
    return (urlsplit(u).hostname or "").lower()

df["host"] = df["url"].map(host_of)
df["tld"] = df["host"].str.rsplit(".", n=1).str[-1]
for label, name in [(0, "legit"), (1, "phishing")]:
    print(f"--- top TLDs: {name}")
    print(df.loc[df.is_phishing == label, "tld"].value_counts().head(8), "\n")

### Sanity check: is the dataset leaking the answer?
Public phishing datasets often have collection artifacts, e.g. every legitimate URL was scraped from a top-sites list with the same prefix. A model can then score ~99% by learning the artifact instead of anything about phishing. Check how the URLs *start* in each class before trusting any score.

In [ ]:
prefix = df["url"].str.extract(r"^(https?://(?:www\.)?)", expand=False).fillna("<no scheme>")
pd.crosstab(prefix, df["is_phishing"].map({0: "legit", 1: "phishing"}), normalize="columns").round(3)

If one prefix shows up almost only in one class, the model will lean on it. Note what you find here; in the next notebook we can strip the scheme and `www.` before featurizing and see how much the score drops. That comparison is a great thing to talk about in an interview.

## 3. Features and split

In [ ]:
%%time
X = featurize(df["url"])
y = df["is_phishing"].values
X.describe().T[["mean", "std", "min", "max"]]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, url_train, url_test = train_test_split(
    X, y, df["url"], test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## 4. Train two baselines

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print("trained", name)

## 5. Evaluate (phishing = positive class)

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             average_precision_score, classification_report)

rows = []
for name, m in models.items():
    proba = m.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "model": name,
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "PR-AUC": average_precision_score(y_test, proba),
        "accuracy": accuracy_score(y_test, pred),
    })
results = pd.DataFrame(rows).set_index("model").round(4)
results

In [ ]:
best_name = results["PR-AUC"].idxmax()
best = models[best_name]
print("best:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=["legit", "phishing"], digits=4))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, display_labels=["legit", "phishing"],
                                      ax=axes[0], colorbar=False)
for name, m in models.items():
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[1])
axes[0].set_title(f"Confusion matrix ({best_name})")
axes[1].set_title("Precision-recall curve")
plt.tight_layout(); plt.show()

### Picking a threshold
0.5 is arbitrary. A security tool usually wants high recall, so find the threshold that keeps recall at 95% or above and see what precision you pay for it.

In [ ]:
from sklearn.metrics import precision_recall_curve
proba = best.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)
target_recall = 0.95
ok = np.where(rec[:-1] >= target_recall)[0]
i = ok[np.argmax(prec[ok])]
print(f"threshold={thr[i]:.3f}  precision={prec[i]:.4f}  recall={rec[i]:.4f}")

## 6. What did the model learn?

In [ ]:
rf = models["random_forest"]
imp = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
imp.plot.barh(figsize=(7, 6), title="Random forest feature importance"); plt.show()

coef = pd.Series(models["logreg"][-1].coef_[0], index=FEATURE_NAMES).sort_values()
coef.plot.barh(figsize=(7, 6), title="Logistic regression coefficients (+ = more phishy)"); plt.show()

## 7. Error analysis
Read the mistakes. This is where ideas for better features (and the deep learning stage) come from.

In [ ]:
errs = pd.DataFrame({"url": url_test.values, "y": y_test, "p_phish": proba})
print("Missed phishing (false negatives), most confident first:")
display(errs[(errs.y == 1) & (errs.p_phish < 0.5)].sort_values("p_phish").head(15))
print("False alarms on legit URLs:")
display(errs[(errs.y == 0) & (errs.p_phish >= 0.5)].sort_values("p_phish", ascending=False).head(15))

## 8. Save the model

In [ ]:
import joblib, json
from phishdetect.data import PROJECT_ROOT
out = PROJECT_ROOT / "models"
joblib.dump(best, out / "baseline.joblib")
(out / "baseline_metrics.json").write_text(results.loc[best_name].to_json(indent=2))
print("saved to", out)

## Results log
Fill this in after running, and copy it into the README:

| model | precision | recall | PR-AUC | notes |
|---|---|---|---|---|
| logreg | | | | |
| random forest | | | | |

**Next notebook:** rerun with the scheme and `www.` stripped (leakage check), then try a char n-gram TF-IDF + logistic regression as a stronger baseline before moving to PyTorch.